# Bimodal: MGD vs REG with lam chosen after the run

Reads `lam_selection.pt` written by `select_lambda.py` (one per beta dir of a sweep
launched with `launch/bimodal_sweep.sh`).

* **lam_selected** — data-driven, no ground truth: the largest lam whose residual
  `Theta(lam) - Theta(0)` still looks like noise (block `E[z^2] <= 1` in every
  t-window; same rule as `turbulence/lamtune_select.ipynb`).
* **lam_oracle** — minimiser of the Fisher-weighted energy error
  `E_runs[(theta_1 - theta*)^T Cov(phi) (theta_1 - theta*)]`, possible only because
  theta* is known here. It checks the data-driven rule; it is not a method.

All quantities are in the solver's phi basis (`phi_a = x^a / a`). "Centered" energy
variance treats the energy up to an additive constant (`tr(C Cov(phi))`, CR bound
`= r / n1`); "uncentered" is the June-2026 definition `E_x[phi^T C phi]`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ── parameters ────────────────────────────────────────────────────────────
RESULTS = Path('results_bim_theta_beta')
# default: the newest sweep that has lam_selection.pt files
_sweeps = sorted(d for d in RESULTS.glob('bim_theta_beta_sweep_*') if any(d.glob('beta_*/lam_selection.pt')))
SWEEP_DIR = Path(os.environ.get('BM_SWEEP_DIR', _sweeps[-1] if _sweeps else ''))
CENTERED = True           # energy up to a constant (see above)

# palette: categorical slots 1-3 for the estimators, gray for the bound
C_MGD, C_SEL, C_ORA, C_REF = '#2a78d6', '#eb6834', '#1baf7a', '#6b6a64'
BLUES = ['#86b6ef', '#6da7ec', '#5598e7', '#3987e5', '#2a78d6', '#256abf', '#1c5cab', '#184f95', '#104281', '#0d366b']
DIVERGING = LinearSegmentedColormap.from_list('z2', ['#1c5cab', '#86b6ef', '#e6e5e0', '#f0a07e', '#b8431a'])
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'lines.linewidth': 2})

files = sorted(SWEEP_DIR.glob('beta_*/lam_selection.pt'))
res = [torch.load(f, map_location='cpu', weights_only=False) for f in files]
res.sort(key=lambda o: o['config']['beta'])
betas = np.array([o['config']['beta'] for o in res])
print('SWEEP_DIR:', SWEEP_DIR, f'({len(res)} betas)')

## Summary per beta

In [ ]:
sfx = '_centered' if CENTERED else ''
def at(o, key, lam):
    return float(o[key][o['lams'].index(lam)]) if lam is not None else np.nan

rows = []
for o in res:
    sel, ora = o['lam_selected'], o['lam_oracle']
    rows.append(dict(beta=o['config']['beta'], K=o['theta_final_reg'].shape[0], lam_sel=sel, lam_oracle=ora,
                     var_mgd=float(o['var_energy_mgd' + sfx]), var_sel=at(o, 'var_energy_reg' + sfx, sel),
                     var_ora=at(o, 'var_energy_reg' + sfx, ora), cr=float(o['cr_bound_energy' + sfx]),
                     mse_mgd=float(o['mse_energy_mgd_centered']), mse_sel=at(o, 'mse_energy_reg_centered', sel),
                     mse_ora=at(o, 'mse_energy_reg_centered', ora),
                     resid=float(o['solve_residual'].max())))
print(f"{'beta':>5} {'K':>4} {'lam_sel':>8} {'lam_orc':>8} | {'Var E: MGD':>11} {'REG sel':>10} {'REG orc':>10} {'CR':>10}"
      f" | {'MSE E: MGD':>11} {'REG sel':>10} {'REG orc':>10} | solve res")
for r in rows:
    print(f"{r['beta']:5.2f} {r['K']:4d} {r['lam_sel'] or 0:8.0e} {r['lam_oracle']:8.0e} | {r['var_mgd']:11.3e} "
          f"{r['var_sel']:10.3e} {r['var_ora']:10.3e} {r['cr']:10.3e} | {r['mse_mgd']:11.3e} {r['mse_sel']:10.3e} "
          f"{r['mse_ora']:10.3e} | {r['resid']:.0e}")

## Energy variance and error vs beta

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
for ax, (key, title) in zip(axes, [('var', 'Var of the energy over runs'),
                                   ('mse', 'Fisher-weighted energy MSE vs theta*')]):
    ax.semilogy(betas, [r[f'{key}_mgd'] for r in rows], 'o-', ms=8, color=C_MGD, label='MGD (raw)')
    ax.semilogy(betas, [r[f'{key}_sel'] for r in rows], 's-', ms=8, color=C_SEL, label='REG, lam selected')
    ax.semilogy(betas, [r[f'{key}_ora'] for r in rows], '^-', ms=8, color=C_ORA, label='REG, lam oracle')
    ax.semilogy(betas, [r['cr'] for r in rows], 'x--', ms=8, color=C_REF, lw=1.5, label='Cramér–Rao')
    ax.set_xlabel('beta'); ax.set_title(title)
axes[0].legend()
plt.tight_layout(); plt.show()

## MSE(lam) per beta

Does the data-driven choice (square) land near the minimum (triangle)? `lam = 0`
is drawn at the left edge; it equals raw MGD up to the solver (same per-node system).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for j, o in enumerate(res):
    lams = np.array(o['lams']); x = np.where(lams > 0, lams, lams[lams > 0].min() / 3)
    mse = o['mse_energy_reg_centered'].numpy()
    col = BLUES[int(round(j * (len(BLUES) - 1) / max(len(res) - 1, 1)))]
    ax.loglog(x, mse, '-', color=col, lw=1.5, label=f"beta={o['config']['beta']:g}")
    if o['lam_selected'] is not None:
        i = o['lams'].index(o['lam_selected']); ax.plot(x[i], mse[i], 's', ms=9, color=col, mec='white', mew=2)
    i = o['lams'].index(o['lam_oracle']); ax.plot(x[i], mse[i], '^', ms=9, color=col, mec='white', mew=2)
ax.set_xlabel('lam  (leftmost point: lam = 0)'); ax.set_ylabel('energy MSE (centered)')
ax.set_title('square: lam selected   triangle: lam oracle')
ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## Selection diagnostics for one beta

In [ ]:
BETA = betas[len(betas) // 2]       # pick a beta
o = res[int(np.argmin(abs(betas - BETA)))]
b = o['blocks'].index(50)
L = [l for l in o['lams'] if l > 0]; idx = [o['lams'].index(l) for l in L]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
im = axes[0].imshow(np.log10(o['z2'][idx, :, b].T), aspect='auto', cmap=DIVERGING, vmin=-1, vmax=1, origin='lower')
axes[0].set_xticks(range(len(L))); axes[0].set_xticklabels([f'{l:.0e}' for l in L], rotation=60)
axes[0].set_yticks(range(len(o['windows']))); axes[0].set_yticklabels([f'[{lo:.2f},{hi:.2f})' for lo, hi, _ in o['windows']])
axes[0].set_title('log10 E[z^2] (w=50): > 0 = lam removes structure'); axes[0].grid(False)
plt.colorbar(im, ax=axes[0])
for j, (lo, hi, _) in enumerate(o['windows']):
    axes[1].loglog(L, o['variance_gain'][idx, j], 'o-', ms=4, lw=1.5,
                   color=BLUES[int(round(j * (len(BLUES) - 1) / max(len(o['windows']) - 1, 1)))], label=f'[{lo:.2f},{hi:.2f})')
for lam, col in [(o['lam_selected'], C_SEL), (o['lam_oracle'], C_ORA)]:
    if lam: axes[1].axvline(lam, color=col, lw=1.5, ls=':')
axes[1].set_xlabel('lam'); axes[1].set_title('across-run variance / variance at lam = 0'); axes[1].legend(fontsize=7)
fig.suptitle(f"beta = {o['config']['beta']:g}"); plt.tight_layout(); plt.show()

## Trajectories for one run

In [ ]:
RUN = 0
t_reg = o['t_reg'].numpy(); tr = o['theta_traj_reg_examples'][RUN].numpy()     # (L, n, r)
labels = ['x', 'x²/2', 'x³/3', 'x⁴/4']
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for a, ax in enumerate(axes.flat):
    ax.plot(t_reg, tr[o['lams'].index(0.0), :, a], color=C_REF, lw=0.6, alpha=0.6, label='lam = 0')
    if o['lam_selected'] is not None:
        ax.plot(t_reg, tr[o['lams'].index(o['lam_selected']), :, a], color=C_SEL, label=f"selected {o['lam_selected']:g}")
    ax.plot(t_reg, tr[o['lams'].index(o['lam_oracle']), :, a], color=C_ORA, ls='--', label=f"oracle {o['lam_oracle']:g}")
    ax.axhline(float(o['target_theta_phi'][a]), color='black', lw=1, ls=':', label='theta*')
    lo, hi = np.percentile(tr[:, :, a], [1, 99]); tgt = float(o['target_theta_phi'][a])
    lo, hi = min(lo, tgt), max(hi, tgt); ax.set_ylim(lo - 0.2 * (hi - lo), hi + 0.2 * (hi - lo))
    ax.set_title(f'coefficient of {labels[a]}')
axes[0, 0].legend(fontsize=7); [ax.set_xlabel('SDE time t') for ax in axes[1]]
fig.suptitle(f"beta = {o['config']['beta']:g}, run {RUN}"); plt.tight_layout(); plt.show()